# 02 - Batch SAM3 Auto-Labeling (text-promptable classes)

Stage L1. Seed-frame SAM3 text prompt + approach-corridor instance pick +
tracker propagation, per sequence. Outputs a label run under
`labels/runs/<run_id>/` (canonical `propagated_boxes.csv` with frame paths
RELATIVE to the shared frame cache, `seed_log.csv`, `failures.csv`, `run.json`).

Config comes from `configs/experiments/labeling_sam3text.yaml`; override any
knob in the config cell. **Validation batch first** (`limit_per_object: 50`),
QA it in notebook 04, then set 0 for the full run. Resumable - rerun the loop
cell any time. **GPU runtime required.**

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import random
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

from vision_uss_research.labeling.boxes_io import (append_boxes, done_keys,
                                                   relativize_frame_path)
from vision_uss_research.labeling.corridor import masks_to_bbox, pick_target_instance
from vision_uss_research.labeling.sam3 import OBJECT_PROMPTS
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import (extract_sequence_frames, frames_window_id,
                                           nearest_frame, select_camera_videos,
                                           with_drive_retry)

CONFIG_NAME = "labeling_sam3text"   # or "labeling_sam3text_v2extra" for the v2 bucket classes
cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments" / f"{CONFIG_NAME}.yaml").read_text())
cfg.update({
    # per-session overrides, e.g.:
    # "limit_per_object": 50,
    # "objects": ["curbstone"],
})
DRY_RUN = False
INVENTORY_RUN = None   # run id under inventory/; None = newest non-smoke run
RUN_ID_OVERRIDE = None # run ids embed the date, so a run resumed on a LATER DAY
                       # would otherwise mint a fresh run dir and start over -
                       # set this to the existing labels/runs/<run_id> to resume it

unknown = [o for o in cfg["objects"] if o not in OBJECT_PROMPTS]
assert not unknown, f"no prompt for: {unknown}"

RUN_ID = RUN_ID_OVERRIDE or new_run_id(cfg["stage"], cfg["tag"])
RUN_DIR = ARTIFACTS / "labels" / "runs" / RUN_ID
FRAMES_ROOT = ARTIFACTS / "frames"
WINDOW_ROOT = FRAMES_ROOT / frames_window_id(cfg["n_frames"], tuple(cfg["ratio_range"]))
BOXES_CSV = RUN_DIR / "propagated_boxes.csv"

inventory_dirs = sorted((ARTIFACTS / "inventory").glob("*"), reverse=True)
inv_dir = (ARTIFACTS / "inventory" / INVENTORY_RUN) if INVENTORY_RUN else next(
    (d for d in inventory_dirs if not d.name.endswith("_smoke")
     and (d / "inventory_folders.csv").exists()), None)
assert inv_dir is not None, "no inventory run found - run notebook 01 first"

record = start_run(RUN_DIR, config={**cfg, "dry_run": DRY_RUN,
                                    "frames_window": WINDOW_ROOT.name},
                   inputs=[f"inventory/{inv_dir.name}"], repo_root=REPO_DIR)
print(f"run dir   : {RUN_DIR}")
print(f"inventory : {inv_dir.name}")

In [ ]:
inv = pd.read_csv(inv_dir / "inventory_folders.csv")
usable = inv[(inv["status"] == "complete")
             & ((inv["has_front_video"] == 1) | (inv["has_rear_video"] == 1))]

rng = random.Random(cfg["seed"])
work = []
for obj in cfg["objects"]:
    mask = usable["label_v2_objects"].fillna("").str.split("|").apply(
        lambda names: any(n.strip().lower() == obj for n in names))
    candidates = sorted(usable[mask]["sequence_id"].tolist())
    if cfg["limit_per_object"]:
        candidates = sorted(rng.sample(candidates,
                                       min(cfg["limit_per_object"], len(candidates))))
    work += [{"sequence_id": s, "target_object": obj} for s in candidates]
    print(f"{obj:<15} {len(candidates)} sequences queued")

done = done_keys(BOXES_CSV)
if done:
    print(f"resuming: {len(done)} sequence/cameras already labeled")
print(f"total queued: {len(work)}")

In [ ]:
if not DRY_RUN:
    from vision_uss_research.labeling.sam3 import Sam3Engine
    engine = Sam3Engine(score_threshold=cfg["score_threshold"])
    print(f"SAM3 loaded on {engine.device}")

## Main labeling loop (resumable - rerun any time)

In [ ]:
import torch

SEED_IDX = int(round((cfg["seed_ratio"] - cfg["ratio_range"][0])
                     / (cfg["ratio_range"][1] - cfg["ratio_range"][0])
                     * (cfg["n_frames"] - 1)))
stats = defaultdict(int)

def log_failure(seq_id, camera, stage, error):
    append_path = RUN_DIR / "failures.csv"
    header = not append_path.exists()
    with append_path.open("a", newline="") as f:
        import csv as _csv
        w = _csv.DictWriter(f, fieldnames=["sequence_id", "camera", "stage", "error"])
        if header:
            w.writeheader()
        w.writerow({"sequence_id": seq_id, "camera": camera,
                    "stage": stage, "error": error})

for item in tqdm(work, desc="Labeling", unit="seq"):
    seq_id, obj = item["sequence_id"], item["target_object"]
    prompt = OBJECT_PROMPTS[obj]
    try:
        videos, _dir = with_drive_retry(select_camera_videos, RAW_ROOT / seq_id)
    except Exception as e:
        log_failure(seq_id, "", "select_camera", f"{type(e).__name__}: {e}")
        stats["failed"] += 1
        continue

    for camera, video_path in videos:
        key = f"{seq_id}::{camera}"
        if key in done:
            stats["skipped_done"] += 1
            continue
        try:
            frames_dir = with_drive_retry(
                extract_sequence_frames, FRAMES_ROOT, seq_id, camera, video_path,
                cfg["n_frames"], tuple(cfg["ratio_range"]))
            if frames_dir is None:
                raise RuntimeError("no frames extracted")
            seed_path = nearest_frame(frames_dir, SEED_IDX)
            if seed_path is None:
                raise RuntimeError("empty frames dir")
            actual_seed_idx = int(seed_path.stem)

            if DRY_RUN:
                stats["dry_ok"] += 1
                continue

            image_rgb = cv2.cvtColor(cv2.imread(str(seed_path)), cv2.COLOR_BGR2RGB)
            result = None
            for attempt in range(2):
                try:
                    result = engine.segment_text(image_rgb, prompt)
                    break
                except torch.cuda.OutOfMemoryError:
                    engine.free()
            if result is None:
                raise RuntimeError("seed segmentation OOM")

            best, diag = pick_target_instance(
                result, image_rgb.shape[:2],
                sigma=cfg["corridor"]["sigma"], min_cy=cfg["corridor"]["min_cy"])
            seed_log = RUN_DIR / "seed_log.csv"
            header = not seed_log.exists()
            with seed_log.open("a", newline="") as f:
                import csv as _csv
                w = _csv.DictWriter(f, fieldnames=["sequence_id", "camera", "target_object",
                                                   "seed_frame_idx", "n_candidates",
                                                   "seed_score", "combined_score"])
                if header:
                    w.writeheader()
                w.writerow({"sequence_id": seq_id, "camera": camera, "target_object": obj,
                            "seed_frame_idx": actual_seed_idx, **diag})
            if best is None:
                stats["no_seed_detection"] += 1
                continue
            seed_box = tuple(map(float, result["boxes"][best]))

            masks = None
            for attempt in range(2):
                try:
                    masks = engine.propagate(frames_dir, actual_seed_idx, {"box": seed_box})
                    break
                except torch.cuda.OutOfMemoryError:
                    engine.free()
            if masks is None:
                raise RuntimeError("propagation OOM")

            rows = []
            for frame_idx, mask in sorted(masks.items()):
                bbox = masks_to_bbox(mask)
                rows.append({
                    "sequence_id": seq_id, "target_object": obj, "camera": camera,
                    "frame_idx": frame_idx,
                    "frame_path": relativize_frame_path(
                        frames_dir / f"{frame_idx:05d}.jpg", WINDOW_ROOT),
                    "x0": bbox[0] if bbox else None, "y0": bbox[1] if bbox else None,
                    "x1": bbox[2] if bbox else None, "y1": bbox[3] if bbox else None,
                    "mask_area_frac": float(mask.mean()),
                })
            append_boxes(BOXES_CSV, rows)
            done.add(key)
            stats["labeled"] += 1
        except Exception as e:
            if not DRY_RUN:
                engine.free()
            log_failure(seq_id, camera, "label", f"{type(e).__name__}: {e}")
            stats["failed"] += 1

print(dict(stats))

## Health check + finish run

In [ ]:
if BOXES_CSV.exists():
    boxes = pd.read_csv(BOXES_CSV)
    health = (boxes.assign(has_box=boxes["x0"].notna())
              .groupby("target_object")
              .agg(sequences=("sequence_id", "nunique"), frames=("has_box", "size"),
                   frames_with_box=("has_box", "sum"),
                   area_frac_median=("mask_area_frac", "median")))
    health["coverage"] = (health["frames_with_box"] / health["frames"]).round(3)
    display(health.round(4))

finish_run(RUN_DIR, summary=dict(stats))
print(f"label run: labels/runs/{RUN_ID}")
print("next: QA this run in notebook 04")